# P5 — Juntándolo todo

Tres modelos, una tarea de recuperación de largo alcance: **GAT** (1 salto, aprendido), el **operador de walk**
(multi-salto, pesos fijos) y **Walk Attention** (multi-salto, aprendido). La teoría predice el resultado; el
código lo confirma.

## 1. Tres operadores sobre el mismo grafo

<img src="../figs-theory/es/T8_three_operators.svg" width="760"/>

## 2. Dos ejes: soporte y pesos

<img src="../figs-theory/es/E5a_support_weights_table.svg" width="760"/>

## 3. Por qué gana o pierde cada modelo

<img src="../figs-theory/es/E5b_why_each.svg" width="760"/>

In [ ]:
# Entrena los tres modelos en la tarea de recuperación del cuello de botella (versión pequeña y rápida).
import torch, torch.nn.functional as F, numpy as np, math
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.transforms import (AttachWalkMasks, collate_walk_masks,
                                   AttachWalkOperators, collate_walk_operators)
from oversquash.models import build_model
K, M, DEPTH = 5, 4, 3   # azar = 1/5 = 0.20
def ds(seed, n=1500):
    g = torch.Generator().manual_seed(seed)
    return [make_bottleneck_graph(K, M, DEPTH, g) for _ in range(n)]

def train_eval(m, tr, va, fwd, epochs=70, lr=2e-3, warmup=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e+1)/warmup if e<warmup
                                            else 0.5*(1+math.cos(math.pi*(e-warmup)/max(1,epochs-warmup))))
    best = 0.0
    for e in range(epochs):
        m.train()
        for b in tr:
            opt.zero_grad(); lg,_ = fwd(m,b)
            F.cross_entropy(lg, b.y, ignore_index=-100).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step(); m.eval(); a=[]
        with torch.no_grad():
            for b in va:
                lg,_ = fwd(m,b); mm=b.root_mask
                a.append((lg[mm].argmax(-1)==b.y[mm]).float().mean().item())
        best = max(best, float(np.mean(a)))
    return best

def run(name, seed=0):
    torch.manual_seed(seed); data = ds(seed); nl = DEPTH+1
    if name=='gat':
        tr=PyGLoader(data[:1200],batch_size=128,shuffle=True); va=PyGLoader(data[1200:],batch_size=128)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None))
    elif name=='walkraw':
        tf=AttachWalkOperators(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_operators)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_operators)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_raw=b.walk_raw)
    else:
        tf=AttachWalkMasks(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_masks)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_masks)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_masks=b.walk_masks)
    m = build_model(name, data[0].x.size(-1), 16, data[0].num_classes, nl, heads=4)
    return train_eval(m, tr, va, fwd)

for name in ['gat','walkraw','walkattn']:
    print(f'{name:10s} accuracy final = {run(name):.3f}')

## 4. El resultado

GAT cae al azar; el operador de walk alcanza pero no selecciona; Walk Attention lo resuelve (1.000 en 5 seeds
en la corrida completa).

<img src="../figs-theory/es/E5c_result.svg" width="760"/>

**El álgebra de caminos de un quiver define una atención sparse multi-salto que mitiga el over-squashing.**